# Project 5: Multiple Linear Regression

**Date**: 2026-09-14  
**Estimated Time**: 12-15 hours  
**Goal**: Extend linear regression to multiple features, detect multicollinearity, engineer an interaction, and compare from-scratch, scikit-learn, and regularized models.

## Learning Objectives

By the end of this project, you will:

- Implement multivariate linear regression from scratch.
- Solve the normal equations for multiple features.
- Train multivariate gradient descent with feature scaling.
- Detect multicollinearity with the Variance Inflation Factor (VIF).
- Engineer and evaluate an interaction feature.
- Compare Ridge, Lasso, and Elastic Net regularization.
- Interpret coefficients and perform residual analysis.

In [ ]:
# Setup: import libraries and load the dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
np.random.seed(42)

# Load the dataset
df = pd.read_csv(r'E:\Machine Learning\datasets\house_prices.csv')
print('Dataset shape:', df.shape)
print('\nFirst 5 rows:')
print(df.head())
print('\nSummary statistics:')
print(df.describe())

## 1. Problem Definition

**Problem**: Predict a house's price using multiple features.

**Type**: Supervised learning - regression.

**Target**: `price`.

**Features**: `size`, `bedrooms`, `age`, `distance`.

**Why this matters**: Real-world house prices depend on more than one factor. Multiple linear regression lets us model the combined effect of many predictors while keeping the model interpretable.

**What makes this challenging**:
- Features can be correlated with each other (multicollinearity).
- Coefficients no longer have the same simple one-feature interpretation.
- Feature scaling is required for gradient descent to converge reliably.
- More features increase the risk of overfitting.

## 2. Mathematical Background

### Hypothesis Function

For `n` features:

h(x) = w0 + w1*x1 + w2*x2 + ... + wn*xn

In matrix form with an added column of ones for the intercept:

h(X) = X @ w

### Cost Function (MSE)

J(w) = (1 / 2m) * sum((X @ w - y) ** 2)

### Gradient of the Cost

∇J(w) = (1 / m) * X.T @ (X @ w - y)

### Batch Gradient Descent Update

w := w - α * ∇J(w)

### Normal Equations (Closed Form)

w = (X.T @ X)^-1 @ X.T @ y

When X.T @ X is not invertible, use the pseudo-inverse.

### Regularization

- **Ridge (L2)**: add λ * sum(w^2) to the cost. Shrinks coefficients but keeps all features.
- **Lasso (L1)**: add λ * sum(|w|) to the cost. Can shrink some coefficients to zero, performing feature selection.
- **Elastic Net**: combination of L1 and L2.

In [ ]:
## 3. Exploratory Data Analysis
# Correlation matrix
corr = df.corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.show()

# Distribution of each feature
feature_cols = ['size', 'bedrooms', 'age', 'distance']
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

for ax, col in zip(axes.flat, feature_cols):
    df[col].hist(bins=20, ax=ax, edgecolor='black')
    ax.set_title(f'Distribution of {col}')
    ax.set_xlabel(col)

plt.tight_layout()
plt.show()

### Interpretation Questions

- Which features are most strongly correlated with price?
- Are any features highly correlated with each other?
- Are any distributions skewed or unusual?

In [ ]:
## 4. Detecting Multicollinearity with VIF

# VIF = 1 / (1 - R^2) where R^2 is from predicting one feature from the others.
# We use scikit-learn to compute the R^2 for each feature against all others.

def compute_vif(data, target_col):
    X = data.drop(columns=[target_col])
    y = data[target_col]
    model = LinearRegression().fit(X, y)
    r2 = model.score(X, y)
    if r2 >= 0.999:
        return float('inf')
    return 1.0 / (1.0 - r2)

vifs = {col: compute_vif(df[feature_cols], col) for col in feature_cols}
vif_df = pd.DataFrame.from_dict(vifs, orient='index', columns=['VIF'])
print(vif_df.round(2))

print('\nRule of thumb: VIF > 5 is moderate, VIF > 10 is strong multicollinearity.')

**What to look for**: High VIF values tell us a feature is predictable from the other features. This inflates coefficient variance and makes interpretation harder.

In [ ]:
## 5. Train/Test Split and Feature Scaling

X = df[feature_cols]
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Keep as DataFrame for convenience
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_cols, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_cols, index=X_test.index)

print(f'Train size: {len(X_train)}')
print(f'Test size:  {len(X_test)}')
print('\nScaled training means (should be near 0):')
print(X_train_scaled.mean().round(4))
print('\nScaled training stds (should be near 1):')
print(X_train_scaled.std().round(4))

In [ ]:
## 6. Baseline Model

# The simplest baseline: predict the mean of the training target for every test sample.
y_pred_baseline = np.full(shape=len(y_test), fill_value=y_train.mean())

mae_base = mean_absolute_error(y_test, y_pred_baseline)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_baseline))

print(f'Baseline (mean) MAE:  {mae_base:.2f}')
print(f'Baseline (mean) RMSE: {rmse_base:.2f}')

In [ ]:
## 7. Multiple Linear Regression from Scratch (Normal Equations)

# Add intercept column
X_train_b = np.hstack([np.ones((X_train_scaled.shape[0], 1)), X_train_scaled.values])
X_test_b = np.hstack([np.ones((X_test_scaled.shape[0], 1)), X_test_scaled.values])

# Closed-form solution using the pseudo-inverse
w_ne = np.linalg.pinv(X_train_b.T @ X_train_b) @ X_train_b.T @ y_train.values

print('Normal equation weights:')
for name, value in zip(['intercept'] + feature_cols, w_ne):
    print(f'  {name:>12s}: {value:>10.2f}')

# Predict on the test set
y_pred_ne = X_test_b @ w_ne

mae_ne = mean_absolute_error(y_test, y_pred_ne)
rmse_ne = np.sqrt(mean_squared_error(y_test, y_pred_ne))
r2_ne = r2_score(y_test, y_pred_ne)

print(f'\nNormal equation -> MAE: {mae_ne:.2f}, RMSE: {rmse_ne:.2f}, R2: {r2_ne:.4f}')

In [ ]:
## 8. Multiple Linear Regression from Scratch (Gradient Descent)

def compute_cost(X, y, w):
    m = len(y)
    return np.sum((X @ w - y) ** 2) / (2 * m)

def gradient_descent(X, y, w, alpha, n_iter):
    m = len(y)
    cost_history = []
    for _ in range(n_iter):
        gradient = (X.T @ (X @ w - y)) / m
        w = w - alpha * gradient
        cost_history.append(compute_cost(X, y, w))
    return w, cost_history

w_init = np.zeros(X_train_b.shape[1])
alpha = 0.1
n_iter = 500

w_gd, costs = gradient_descent(
    X_train_b, y_train.values, w_init, alpha, n_iter
)

print(f'GD final cost: {costs[-1]:.2f}')
print('GD final weights:')
for name, value in zip(['intercept'] + feature_cols, w_gd):
    print(f'  {name:>12s}: {value:>10.2f}')

plt.figure(figsize=(8, 4))
plt.plot(costs)
plt.xlabel('Iteration')
plt.ylabel('Cost')
plt.title('Gradient Descent Convergence')
plt.show()

y_pred_gd = X_test_b @ w_gd
mae_gd = mean_absolute_error(y_test, y_pred_gd)
rmse_gd = np.sqrt(mean_squared_error(y_test, y_pred_gd))
r2_gd = r2_score(y_test, y_pred_gd)

print(f'\nGradient descent -> MAE: {mae_gd:.2f}, RMSE: {rmse_gd:.2f}, R2: {r2_gd:.4f}')

In [ ]:
## 9. Scikit-Learn Implementation

model = LinearRegression()
model.fit(X_train_scaled, y_train)
y_pred_sk = model.predict(X_test_scaled)

print(f'Scikit-Learn intercept: {model.intercept_:.2f}')
print('Scikit-Learn coefficients:')
for name, value in zip(feature_cols, model.coef_):
    print(f'  {name:>12s}: {value:>10.2f}')

mae_sk = mean_absolute_error(y_test, y_pred_sk)
rmse_sk = np.sqrt(mean_squared_error(y_test, y_pred_sk))
r2_sk = r2_score(y_test, y_pred_sk)

print(f'\nScikit-Learn -> MAE: {mae_sk:.2f}, RMSE: {rmse_sk:.2f}, R2: {r2_sk:.4f}')

**Check**: The from-scratch and scikit-learn weights and metrics should be nearly identical. Small differences come from numerical precision.

In [ ]:
## 10. Feature Engineering: Add an Interaction Term

# The effect of size may depend on the number of bedrooms.
X_train_int = X_train.copy()
X_test_int = X_test.copy()

X_train_int['size_x_bedrooms'] = X_train_int['size'] * X_train_int['bedrooms']
X_test_int['size_x_bedrooms'] = X_test_int['size'] * X_test_int['bedrooms']

scaler_int = StandardScaler()
X_train_int_scaled = scaler_int.fit_transform(X_train_int)
X_test_int_scaled = scaler_int.transform(X_test_int)

model_int = LinearRegression()
model_int.fit(X_train_int_scaled, y_train)
y_pred_int = model_int.predict(X_test_int_scaled)

mae_int = mean_absolute_error(y_test, y_pred_int)
rmse_int = np.sqrt(mean_squared_error(y_test, y_pred_int))
r2_int = r2_score(y_test, y_pred_int)

print(f'With interaction -> MAE: {mae_int:.2f}, RMSE: {rmse_int:.2f}, R2: {r2_int:.4f}')
print('\nInteraction model coefficients:')
for name, value in zip(list(X_train_int.columns) + ['intercept'], list(model_int.coef_) + [model_int.intercept_]):
    print(f'  {name:>18s}: {value:>10.2f}')

**Interpretation**: Did the interaction term improve the model? If so, by how much? Is the improvement meaningful given the small dataset?

In [ ]:
## 11. Regularization: Ridge, Lasso, and Elastic Net

for Model, name in [(Ridge, 'Ridge'), (Lasso, 'Lasso'), (ElasticNet, 'ElasticNet')]:
    m = Model(alpha=1.0, max_iter=10000, random_state=42)
    m.fit(X_train_scaled, y_train)
    y_pred = m.predict(X_test_scaled)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f'{name:12s} -> MAE: {mae:.2f}, RMSE: {rmse:.2f}, R2: {r2:.4f}')
    print(f'  Coefficients: {dict(zip(feature_cols, m.coef_.round(2)))}')
    print()

**Observation**: Lasso may drive some coefficients to exactly zero, demonstrating feature selection. Ridge shrinks all coefficients smoothly.

In [ ]:
## 12. Residual Analysis

residuals = y_test.values - y_pred_sk

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_pred_sk, residuals, alpha=0.7)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Predicted Price')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted')

axes[1].hist(residuals, bins=15, edgecolor='black')
axes[1].set_xlabel('Residual')
axes[1].set_title('Distribution of Residuals')

plt.tight_layout()
plt.show()

print(f'Mean residual: {residuals.mean():.2f}')
print(f'Std residual:  {residuals.std():.2f}')

In [ ]:
## 13. Summary of Results

results = pd.DataFrame({
    'Model': ['Baseline (mean)', 'Normal Eq (scratch)', 'Gradient Descent (scratch)', 'Scikit-Learn', 'With interaction', 'Ridge', 'Lasso', 'ElasticNet'],
    'MAE': [mae_base, mae_ne, mae_gd, mae_sk, mae_int, None, None, None],
    'RMSE': [rmse_base, rmse_ne, rmse_gd, rmse_sk, rmse_int, None, None, None],
    'R2': [np.nan, r2_ne, r2_gd, r2_sk, r2_int, np.nan, np.nan, np.nan]
})

# Fill in regularization values from the loop
ridge = Ridge(alpha=1.0).fit(X_train_scaled, y_train)
y_pred_ridge = ridge.predict(X_test_scaled)
lasso = Lasso(alpha=1.0, max_iter=10000).fit(X_train_scaled, y_train)
y_pred_lasso = lasso.predict(X_test_scaled)
enet = ElasticNet(alpha=1.0, max_iter=10000, random_state=42).fit(X_train_scaled, y_train)
y_pred_enet = enet.predict(X_test_scaled)

results.loc[results['Model'] == 'Ridge', 'MAE'] = mean_absolute_error(y_test, y_pred_ridge)
results.loc[results['Model'] == 'Ridge', 'RMSE'] = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
results.loc[results['Model'] == 'Ridge', 'R2'] = r2_score(y_test, y_pred_ridge)

results.loc[results['Model'] == 'Lasso', 'MAE'] = mean_absolute_error(y_test, y_pred_lasso)
results.loc[results['Model'] == 'Lasso', 'RMSE'] = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
results.loc[results['Model'] == 'Lasso', 'R2'] = r2_score(y_test, y_pred_lasso)

results.loc[results['Model'] == 'ElasticNet', 'MAE'] = mean_absolute_error(y_test, y_pred_enet)
results.loc[results['Model'] == 'ElasticNet', 'RMSE'] = np.sqrt(mean_squared_error(y_test, y_pred_enet))
results.loc[results['Model'] == 'ElasticNet', 'R2'] = r2_score(y_test, y_pred_enet)

results = results.round(2)
print(results)

## 14. Conclusions and Next Steps

**What we learned**:
- Multiple linear regression extends simple linear regression to any number of features.
- The normal equation gives an exact solution but can be slow for very large datasets.
- Feature scaling makes gradient descent stable and fast.
- VIF helps detect multicollinearity before it distorts coefficients.
- Interaction features can capture combined effects, but should be validated on a test set.
- Regularization controls model complexity and can improve generalization.

**Questions to ask before moving on**:

1. Which model performed best on the test set?
2. Did the interaction term help? Why or why not?
3. Were any features collinear? How did you handle it?
4. How would you scale this to thousands of features?

**Next project**: Polynomial Regression — where we model non-linear relationships.